# Generate stop_times.txt

Generates GTFS stop_times file with travel times based on route speeds and dwell times.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import geopandas as gpd
from gtfs_common import map_route_params


## Parameters

In [2]:
# Path to params.json (same directory as this notebook)
_params_path = "../params.json"
with open(_params_path, encoding="utf-8") as f:
    p = json.load(f)

In [3]:
CITY = p["city"]
dwell_time_station_minutes = p["stop_times"]["dwell_time_station_minutes"]

# Load speeds configured by route
speed_by_route = p["stop_times"].get("speed_by_route", {})
speed_global = speed_by_route.get("default", 20.13)

print(f"Global speed: {speed_global} km/h")
print(f"Routes with specific speed: {len([k for k in speed_by_route.keys() if k != 'default'])}")

Global speed: 20.13 km/h
Routes with specific speed: 5


In [5]:
# --- GTFS folder ---
OUTPUT_DIR_gtfs = Path(f"../data/{CITY}/gtfs-frequencies")
OUTPUT_DIR_gtfs.mkdir(parents=True, exist_ok=True)
print(f"Input: {OUTPUT_DIR_gtfs.absolute()}")

# --- Processed routes folder ---
OUTPUT_DIR_processed = Path(f"../data/{CITY}/processed")
OUTPUT_DIR_processed.mkdir(parents=True, exist_ok=True)
print(f"Input: {OUTPUT_DIR_processed.absolute()}")

Input: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/gtfs-frequencies
Input: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/processed


## Read files

In [ ]:
routes_df = pd.read_csv(OUTPUT_DIR_gtfs / "routes.txt")
speed_by_route_id = map_route_params(
    routes_df,
    speed_by_route,
    speed_global,
    kind="speed_kmh",
)
print(f"Speed configuration covers {len(speed_by_route_id)} routes")


In [ ]:
segments_gdf = gpd.read_file(OUTPUT_DIR_processed / "segments.geojson")
print(f"Loaded {len(segments_gdf)} segments")

missing = sorted(set(segments_gdf["route_name"]) - set(speed_by_route_id))
if missing:
    raise ValueError(f"No speed for routes: {missing}")

speed_configured = {
    route_name: speed_by_route_id[route_name]
    for route_name in segments_gdf["route_name"].unique()
}
segments_gdf.head()


## Calculate time to travel each segment

In [10]:
from pyproj import CRS
import numpy as np

def _utm_from_centroid(gdf: gpd.GeoDataFrame) -> CRS:
    g4326 = gdf.to_crs(4326)
    c = g4326.union_all().centroid
    lon, lat = float(c.x), float(c.y)
    zone = int((lon + 180) // 6) + 1
    epsg = 32600 + zone if lat >= 0 else 32700 + zone
    return CRS.from_epsg(epsg)

def _crs_is_metric(crs: CRS) -> bool:
    try:
        return any(ai.unit_name.lower().startswith(("metre", "meter")) for ai in crs.axis_info)
    except Exception:
        return False

In [11]:
def add_time_kmh_min(
    segments_gdf: gpd.GeoDataFrame,
    speed_kmh,                 # float | dict | pd.Series (by route) | pd.Series (by row)
    by: str = "route_name",      # column for mapping speed when dict/Series by route
    length_col: str = "length_m",
    inplace: bool = False
) -> gpd.GeoDataFrame:
    """
    Adds ONLY:
      - speed_kmh  (km/h)
      - time_min   (minutes)
    """
    if "geometry" not in segments_gdf.columns:
        raise ValueError("segments_gdf must have 'geometry'.")

    g = segments_gdf if inplace else segments_gdf.copy()

    # 1) Length in meters (if not exists, calculate it)
    if length_col in g.columns:
        len_m = g[length_col].to_numpy(dtype=float)
    else:
        if g.crs is None:
            raise ValueError("The GeoDataFrame has no CRS (e.g., EPSG:4326).")
        crs_in = CRS.from_user_input(g.crs)
        crs_m = g.crs if _crs_is_metric(crs_in) else _utm_from_centroid(g)
        len_m = g.to_crs(crs_m).geometry.length.to_numpy()
        g[length_col] = len_m

    # 2) Resolve speed (km/h) vectorized
    if np.isscalar(speed_kmh):
        v_kmh = np.full(len(g), float(speed_kmh), dtype=float)
    elif isinstance(speed_kmh, dict):
        v_kmh = pd.Series(speed_kmh).reindex(g[by]).to_numpy(dtype=float)
    elif isinstance(speed_kmh, pd.Series):
        v_kmh = (speed_kmh.to_numpy(dtype=float) if speed_kmh.index.equals(g.index)
                 else speed_kmh.reindex(g[by]).to_numpy(dtype=float))
    else:
        raise TypeError("speed_kmh must be float, dict, or pd.Series")

    if np.any(~np.isfinite(v_kmh)) or np.any(v_kmh <= 0):
        raise ValueError("Invalid km/h speeds (missing or <= 0).")

    # 3) Time in minutes (without intermediate columns)
    # time_min = (dist_km / kmh) * 60 = (len_m/1000) * 60 / v_kmh
    g["speed_kmh"] = v_kmh
    g["time_min_travel"]  = (g[length_col].to_numpy(dtype=float) / 1000.0) * 60.0 / v_kmh
    return g

In [12]:
# Use existing function with configuration by route_name
segments_with_time = add_time_kmh_min(
    segments_gdf, 
    speed_kmh=speed_configured,
    by="route_name"
)

segments_with_time["dwell_time"] = dwell_time_station_minutes

segments_with_time["total_time"] = segments_with_time["time_min_travel"]  + segments_with_time["dwell_time"] 

/Users/danielbustillos/miniconda3/envs/analisis-general/lib/python3.10/site-packages/shapely/set_operations.py:421: RuntimeWarning: invalid value encountered in unary_union
  return lib.unary_union(collections, **kwargs)


In [13]:
# Show speed summary by route
print("Speed summary by route:")
speed_summary = segments_with_time.groupby("route_name")["speed_kmh"].first().reset_index()

speed_summary.head(10)

Speed summary by route:


,route_name,speed_kmh
0,100_52 Norte Villas La Hacienda R-1,20.13
1,112_Mulsay Juan Pablo Ii,20.13
2,114_Carranza,20.13
3,115_San Lucas,20.13
4,11_50 Penal Paso Texas,20.13
5,121_Pensiones,20.13
6,"124_Chichí Suárez, Sitpach",20.13
7,126_63 Periferico-San Camilo,20.13
8,128_67 Mulsay R-1,20.13
9,132_79 Aviación,20.13


In [14]:
segments_with_time.head(5)

,route_name,shape_id,segment_seq,segment_id,from_stop_id,to_stop_id,from_measure_m,to_measure_m,geometry,length_m,speed_kmh,time_min_travel,dwell_time,total_time
0,2_42 Caseta,shape_42 Caseta,0,Seg_2_42 Caseta_0000,2_42 Caseta_0000,2_42 Caseta_0001,0.0,200.0,"LINESTRING (-89.62102 20.96196, -89.62276 20.9...",197.839049,20.13,0.589684,0.2,0.789684
1,2_42 Caseta,shape_42 Caseta,1,Seg_2_42 Caseta_0001,2_42 Caseta_0001,2_42 Caseta_0002,200.0,400.0,"LINESTRING (-89.62278 20.96209, -89.62295 20.9...",197.839484,20.13,0.589685,0.2,0.789685
2,2_42 Caseta,shape_42 Caseta,2,Seg_2_42 Caseta_0002,2_42 Caseta_0002,2_42 Caseta_0003,400.0,600.0,"LINESTRING (-89.62201 20.96102, -89.62162 20.9...",197.838461,20.13,0.589682,0.2,0.789682
3,2_42 Caseta,shape_42 Caseta,3,Seg_2_42 Caseta_0003,2_42 Caseta_0003,2_42 Caseta_0004,600.0,800.0,"LINESTRING (-89.62018 20.96051, -89.61837 20.9...",197.837288,20.13,0.589679,0.2,0.789679
4,2_42 Caseta,shape_42 Caseta,4,Seg_2_42 Caseta_0004,2_42 Caseta_0004,2_42 Caseta_0005,800.0,1000.0,"LINESTRING (-89.61837 20.95998, -89.61699 20.9...",197.836116,20.13,0.589675,0.2,0.789675


## Build stop times

In [15]:
def sec_to_gtfs_time(sec):
    """Converts seconds from midnight to HH:MM:SS format."""
    h = int(sec) // 3600
    m = (int(sec) % 3600) // 60
    s = int(sec) % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

dwell_sec = dwell_time_station_minutes * 60

rows = []
for (route_name, shape_id), grp in segments_with_time.sort_values("segment_seq").groupby(["route_name", "shape_id"]):
    grp = grp.sort_values("segment_seq").reset_index(drop=True)
    trip_id = f"{route_name}_trip_00"
    current_time_sec = 0.0

    # First stop (origin of first segment)
    initial_stop_id = grp.iloc[0]["from_stop_id"]
    rows.append({
        "trip_id": trip_id,
        #"route_name": route_name,
        "timepoint": 1,
        "stop_id": initial_stop_id,
        "stop_sequence": 1,
        "arrival_time": sec_to_gtfs_time(0),
        "departure_time": sec_to_gtfs_time(dwell_sec),
    })
    current_time_sec = dwell_sec
    seq = 2

    # Rest of stops (to_stop_id of each segment)
    for i, row in grp.iterrows():
        current_time_sec += row["time_min_travel"] * 60
        arrival = sec_to_gtfs_time(current_time_sec)
        current_time_sec += dwell_sec
        departure = sec_to_gtfs_time(current_time_sec)
        rows.append({
            "trip_id": trip_id,
            #"route_name": route_name,
            "timepoint": 1,
            "stop_id": row["to_stop_id"],
            "stop_sequence": seq,
            "arrival_time": arrival,
            "departure_time": departure,
        })
        seq += 1

stop_times = pd.DataFrame(rows)

In [16]:
stop_times.head()

,trip_id,timepoint,stop_id,stop_sequence,arrival_time,departure_time
0,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0000,1,00:00:00,00:00:12
1,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0001,2,00:00:47,00:00:59
2,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0002,3,00:01:34,00:01:46
3,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0003,4,00:02:22,00:02:34
4,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0004,5,00:03:09,00:03:21


## Export

In [17]:
stop_times.to_csv(OUTPUT_DIR_gtfs / "stop_times.txt", index=False)